# Support Vector Regression

Standard tabular ML workflow for LD50 regression.


In [11]:
from pathlib import Path
import sys
import warnings

import numpy as np

PROJECT_ROOT = Path("../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from sklearn.svm import SVR as SklearnSVR

from utils.modeling import artifact_paths, load_tabular_arrays, save_ml_run


In [12]:
MODEL_NAME = "svm"
BASE_DIR = Path.cwd()
RANDOM_STATE = 42
MAX_TRAIN_ROWS = 50_000

X_train, X_valid, X_test, y_train, y_valid, y_test, feature_names = load_tabular_arrays('../../data/feature_selection')
X_train = X_train.astype(np.float32)
X_valid = X_valid.astype(np.float32)
X_test = X_test.astype(np.float32)

paths = artifact_paths(
    BASE_DIR,
    MODEL_NAME,
    model_suffix=".pkl",
    include_importance=True,
)


In [13]:
try:
    from cuml.svm import SVR as CumlSVR

    model = CumlSVR(kernel="rbf", C=10, epsilon=0.1, gamma="scale")
except Exception:
    warnings.warn("cuML SVR not available; using sklearn SVR on CPU.")
    model = SklearnSVR(kernel="rbf", C=10, epsilon=0.1, gamma="scale")

if X_train.shape[0] > MAX_TRAIN_ROWS:
    sample_idx = np.random.RandomState(RANDOM_STATE).choice(
        X_train.shape[0],
        size=MAX_TRAIN_ROWS,
        replace=False,
    )
    X_fit = X_train[sample_idx]
    y_fit = y_train[sample_idx]
else:
    X_fit = X_train
    y_fit = y_train

model.fit(X_fit, y_fit)
predictions = model.predict(X_test)

C:\Users\Tommaso\AppData\Local\Temp\ipykernel_23044\2230096679.py:6: UserWarning: cuML SVR not available; using sklearn SVR on CPU.
  warnings.warn("cuML SVR not available; using sklearn SVR on CPU.")


In [14]:
metrics = save_ml_run("SVM", model, paths, X_test, y_test, predictions, feature_names)

[SVM] RMSE: 0.6648 | MAE: 0.4840 | R2: 0.5053
